# Medidas de Heterogeneidad

### Preprocesamiento de DataFrame

In [1]:
import sys
from pathlib import Path
import polars as pl
import numpy as np

#Para poder utilizar archivos del proyecto
ROOT = Path.cwd().parent 
if str(ROOT) not in sys.path:
    sys.path.insert(0, str(ROOT))
from src.cleaning.preprocessing import pipeline_preprocesamiento

df = pipeline_preprocesamiento()

### Coeficiente de Variación (CV)

In [2]:
def calcular_cv(df: pl.DataFrame, col: str, peso_col: str | None = None) -> float:
    """Calcula el Coeficiente de Variación (CV = desv_estandar / media).
    Soporta cálculo ponderado si se proporciona peso_col.
    """
    df_clean = df.filter(pl.col(col).is_not_null())
    
    if peso_col:
        df_clean = df_clean.filter(pl.col(peso_col).is_not_null())
        val = df_clean[col].to_numpy()
        w = df_clean[peso_col].to_numpy()
        
        media = np.average(val, weights=w)
        varianza = np.average((val - media) ** 2, weights=w)
        std = np.sqrt(varianza)
    else:
        media = df_clean[col].mean()
        std = df_clean[col].std()

    if media == 0 or media is None:
        return 0.0
    return float(std / media)

### Índice de Gini-Simpson e IQV

In [3]:
def calcular_gini_simpson_iqv(p: pl.Series) -> tuple[float, float]:
    """Calcula el Índice de Gini-Simpson y el Índice de Variación Cualitativa (IQV).
    Recibe una serie de Polars con las proporciones p_i.
    """
    p_array = p.to_numpy()
    k = len(p_array)

    gini_simpson = float(1.0 - np.sum(p_array ** 2))
    iqv = float((k / (k - 1)) * gini_simpson) if k > 1 else 0.0

    return gini_simpson, iqv


### Entropía de Shannon

In [4]:
def calcular_entropia_shannon(p: pl.Series, base: float = 2.0) -> tuple[float, float]:
    """Calcula la Entropía de Shannon (bits) y la Entropía Normalizada (0 a 1).
    Recibe una serie de Polars con las proporciones p_i.
    """
    p_array = p.to_numpy()
    p_valid = p_array[p_array > 0]
    k = len(p_array)

    shannon_bits = float(-np.sum(p_valid * np.log2(p_valid)))
    shannon_norm = float(shannon_bits / np.log2(k)) if k > 1 else 0.0

    return shannon_bits, shannon_norm

### Procesamiento Integrador

In [5]:

def procesar_heterogeneidad_categorica(
    df: pl.DataFrame, 
    columnas: list[str], 
    peso_col: str | None = "factor_expansion"
) -> pl.DataFrame:
    """Procesa una lista de columnas categóricas e integra los resultados en un DataFrame."""
    resultados = []

    for col in columnas:
        cond = pl.col(col).is_not_null()
        if peso_col:
            cond = cond & pl.col(peso_col).is_not_null()

        df_filtrado = df.filter(cond)

        # Obtención de frecuencias relativas (p_i)
        if peso_col:
            frecuencias = df_filtrado.group_by(col).agg(
                peso_sum=pl.col(peso_col).sum()
            ).select(
                p=pl.col("peso_sum") / pl.col("peso_sum").sum()
            )
        else:
            frecuencias = df_filtrado.group_by(col).len().select(
                p=pl.col("len") / pl.col("len").sum()
            )

        k = frecuencias.height
        proporciones = frecuencias["p"]

        # Funciones varias
        gini_simpson, iqv = calcular_gini_simpson_iqv(proporciones)
        shannon_bits, shannon_norm = calcular_entropia_shannon(proporciones)

        resultados.append({
            "Atributo": col,
            "Categorías (K)": k,
            "Gini-Simpson": round(gini_simpson, 6),
            "Variación Cualitativa (IQV)": round(iqv, 6),
            "Entropía Shannon (bits)": round(shannon_bits, 6),
            "Entropía Normalizada": round(shannon_norm, 6)
        })

    return pl.DataFrame(resultados)

### Cálculo de medidas de heterogeneidad

In [6]:
columnas_cat = ["nivel_escolaridad", "nom_municipio", "tiene_ahorros_id"]
tabla_medidas = procesar_heterogeneidad_categorica(df, columnas_cat, peso_col="factor_expansion")

print("Tabla de Heterogeneidad Categórica")
print(tabla_medidas)

Tabla de Heterogeneidad Categórica
shape: (3, 6)
┌───────────────────┬─────────────────┬──────────────┬─────────────┬───────────┬─────────────┐
│ Atributo          ┆ Categorías (K)  ┆ Gini-Simpson ┆ Variación   ┆ Entropía  ┆ Entropía    │
│ ---               ┆ ---             ┆ ---          ┆ Cualitativa ┆ Shannon   ┆ Normalizada │
│ str               ┆ i64             ┆ f64          ┆ (IQV)       ┆ (bits)    ┆ ---         │
│                   ┆                 ┆              ┆ ---         ┆ ---       ┆ f64         │
│                   ┆                 ┆              ┆ f64         ┆ f64       ┆             │
╞═══════════════════╪═════════════════╪══════════════╪═════════════╪═══════════╪═════════════╡
│ nivel_escolaridad ┆ 6               ┆ 0.665619     ┆ 0.798743    ┆ 1.982537  ┆ 0.76695     │
│ nom_municipio     ┆ 1206            ┆ 0.995936     ┆ 0.996763    ┆ 8.919625  ┆ 0.871396    │
│ tiene_ahorros_id  ┆ 2               ┆ 0.191773     ┆ 0.383546    ┆ 0.492106  ┆ 0.492106    │
└